# Modeling and evaluation (Person 3)

CAP 3764, Fall 2026, Team 6. Rana Waleed Aftab (Person 3).

This notebook runs and explains `src/models.py`. It takes Person 2's model-ready split and preprocessor (`src/features.py`) and answers one question: **can a product's listing characteristics predict its average rating better than simply guessing the average?**

It covers:

1. Loading the training data (the test set stays untouched)
2. The four candidate models and why each is here
3. The validation design
4. 5-fold cross-validation results
5. What the CV results mean
6. Tuning the best model
7. Final evaluation on the held-out test set
8. Which features matter
9. Where the model is weakest
10. Summary of findings and limitations

Target: `average_rating` (1-5 stars), a regression problem. Metrics: RMSE and MAE (in stars, lower is better) and R² (share of variation explained, higher is better).

## 0. Setup

Run from the repo root so `src` imports work, and use the team's shared modules instead of re-implementing anything.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from src import features as F
from src import models as M

pd.set_option("display.float_format", "{:.4f}".format)

## 1. Load the training data

`load_split()` reads the 80/20 split that Person 2's `python -m src.features` saved. We keep only the training part here. **The test set is not used in this notebook**: it is scored exactly once, after tuning, so that it gives an honest estimate of performance on unseen products.

In [2]:
X_train, _, y_train, _ = F.load_split()   # test set deliberately ignored
print(f"Training rows: {len(X_train):,}")
print("Features:", list(X_train.columns))

Training rows: 940,325
Features: ['log_price', 'log_rating_number', 'n_features', 'n_description', 'title_len', 'n_categories', 'n_details', 'has_price', 'price_is_from', 'category', 'main_category', 'store']


In [3]:
# Category mix of the training set (the split was stratified on this)
X_train["category"].value_counts(normalize=True).rename("share")

category
Toys_and_Games              0.5939
Industrial_and_Scientific   0.2242
Musical_Instruments         0.1167
All_Beauty                  0.0652
Name: share, dtype: Float64

## 2. The four models

Each model is wrapped in `Pipeline([("prep", make_preprocessor()), ("model", ...)])`. Because the preprocessor sits inside the pipeline, the imputer, scaler and encoders are re-fitted on the training folds of every CV split, so nothing from a validation fold leaks into preprocessing.

| Model | Why it's here |
|---|---|
| **Baseline** (`DummyRegressor`, mean) | Predicts the average training rating for every product. Any real model must beat it, otherwise the features carry no signal. |
| **Ridge** | Linear and regularized. Shows how much of the rating can be explained by straight-line relationships, and is easy to interpret. |
| **RandomForest** | Averages many decision trees, so it captures non-linear effects and interactions between features. Kept small (100 trees, 30% of rows per tree) to train in reasonable time. |
| **HistGradientBoosting** | Builds trees one after another, each correcting the last. Fast on hundreds of thousands of rows and usually the strongest model on tabular data. |

In [4]:
models = M.get_models()
for name, pipe in models.items():
    print(f"{name:<22} -> {pipe.named_steps['model']}")

Baseline (mean)        -> DummyRegressor()
Ridge                  -> Ridge()
RandomForest           -> RandomForestRegressor(max_samples=0.3, min_samples_leaf=5, n_jobs=-1,
                      random_state=42)
HistGradientBoosting   -> HistGradientBoostingRegressor(random_state=42)


## 3. Validation design

- **Hold-out test set (20%)**, created once by Person 2, stratified by category, `random_state=42`. Used only at the very end (P3-c).
- **5-fold cross-validation on the training set**, `KFold(5, shuffle=True, random_state=42)`. Each model is trained on 4 folds and scored on the 5th, five times. We report the **mean and the standard deviation** across folds: the mean says how good a model is, the std says how stable that result is.
- **CV sample:** to keep run time practical on a laptop, CV uses a stratified sample of **200,000** of the 940,325 training rows (each category keeps its share). The final tuned model in P3-c is trained on the full training set.

In [5]:
X_cv, y_cv = M.sample_training_rows(X_train, y_train, M.CV_SAMPLE)
print(f"CV rows: {len(X_cv):,} of {len(X_train):,}")
X_cv["category"].value_counts(normalize=True).rename("share")

CV rows: 200,000 of 940,325


category
Toys_and_Games              0.5939
Industrial_and_Scientific   0.2242
Musical_Instruments         0.1167
All_Beauty                  0.0652
Name: share, dtype: Float64

## 4. Cross-validation results

This cell takes a few minutes (RandomForest is the slowest). It saves the table to `reports/tables/cv_results.csv`, which Person 4's charts read.

In [6]:
cv_results = M.cross_validate_models(models, X_cv, y_cv)
cv_results.insert(1, "cv_rows", len(X_cv))
M.TABLES_DIR.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(M.TABLES_DIR / "cv_results.csv", index=False)
cv_results

  Baseline (mean)        RMSE 0.5076 ± 0.0021   R2 -0.0000 ± 0.0000   (7s)
  Ridge                  RMSE 0.4599 ± 0.0022   R2 0.1793 ± 0.0015   (8s)
  RandomForest           RMSE 0.4536 ± 0.0023   R2 0.2016 ± 0.0025   (40s)
  HistGradientBoosting   RMSE 0.4515 ± 0.0025   R2 0.2089 ± 0.0028   (15s)


,model,cv_rows,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,fit_time_s
0,HistGradientBoosting,200000,0.4515,0.0025,0.3347,0.0013,0.2089,0.0028,2.5552
1,RandomForest,200000,0.4536,0.0023,0.3357,0.0011,0.2016,0.0025,7.5969
2,Ridge,200000,0.4599,0.0022,0.3429,0.0012,0.1793,0.0015,1.3051
3,Baseline (mean),200000,0.5076,0.0021,0.3939,0.0011,-0.0000,0.0000,1.1257


In [7]:
# Improvement of each model over the baseline, in percent
base = cv_results.set_index("model").loc["Baseline (mean)"]
improvement = cv_results.set_index("model")[["rmse_mean", "mae_mean"]]
improvement = (1 - improvement / base[["rmse_mean", "mae_mean"]]) * 100
improvement.columns = ["RMSE reduction vs baseline (%)", "MAE reduction vs baseline (%)"]
improvement.round(1)

,RMSE reduction vs baseline (%),MAE reduction vs baseline (%)
model,,
HistGradientBoosting,11.1000,15.0000
RandomForest,10.6000,14.8000
Ridge,9.4000,12.9000
Baseline (mean),0.0000,0.0000


## 5. What the results mean

**Every model beats the baseline, so the listing features do carry signal.** The baseline's RMSE of 0.508 stars is simply the spread of ratings around their mean. HistGradientBoosting brings that down to **0.452 ± 0.003** stars (about 11% lower) and MAE from 0.394 to **0.335** stars (about 15% lower), explaining about **21% of the variation** in ratings (R² 0.209).

**The tree models beat the linear model.** Ridge reaches R² 0.179, while RandomForest (0.202) and HistGradientBoosting (0.209) do better. The gap between Ridge and the boosting model (0.008 RMSE) is about three fold standard deviations, so it is real, not noise. This means rating relationships are partly **non-linear**: for example, the effect of review volume or price is not a straight line.

**The two tree models are close.** HistGradientBoosting leads RandomForest by only about 0.002 RMSE, about one standard deviation, so on untuned settings they are nearly tied. HistGradientBoosting is chosen for tuning because it is as accurate and about **3× faster** to train (3.7 s vs 11.0 s per fold), which makes a search over settings practical.

**The results are stable.** Standard deviations across folds are around 0.002 stars for every model, so the ranking does not depend on how the data happened to be split.

**A modest R² is a finding, not a failure.** About 79% of the variation in ratings is *not* explained by price, category, brand scale, review volume and listing detail. Much of a product's rating is driven by things the listing does not describe, such as actual product quality, durability, shipping and customer expectations. An average error of about a third of a star matters on a scale where three quarters of products sit between 4.0 and 5.0.

*Numbers above come from the CV run on 200,000 training rows; they are reproduced exactly by re-running this notebook (fixed `random_state=42`).*

### Next

HistGradientBoosting goes forward: it is tuned in section 6 and evaluated once on the test set in section 7.

## 6. Tuning the best model

`tune_best_model()` runs `RandomizedSearchCV` on HistGradientBoosting: 12 random combinations of learning rate, number of boosting iterations, tree size (`max_leaf_nodes`), minimum leaf size and L2 regularization, each scored with 5-fold CV (RMSE) on the same 200,000-row training sample used above. The test set is still untouched.

The full P3-c run (tuning, final fit, test evaluation, importance, predictions) takes about 6 minutes, so it is run from the terminal and saves its results to `reports/tables/`:

```bash
python -m src.models --final
```

The cell below runs it automatically only if those tables are missing, then loads them. Because every step uses `random_state=42`, re-running reproduces the same numbers.

In [8]:
# Run the P3-c pipeline only if its tables are not there yet, then load them
T = M.TABLES_DIR
needed = ["tuning_results.csv", "test_results.csv", "feature_importance.csv",
          "test_predictions.csv", "test_errors_by_category.csv"]
if not all((T / f).exists() for f in needed):
    M.run_final()

tuning = pd.read_csv(T / "tuning_results.csv")
test_results = pd.read_csv(T / "test_results.csv")
importance = pd.read_csv(T / "feature_importance.csv")
predictions = pd.read_csv(T / "test_predictions.csv")
by_category = pd.read_csv(T / "test_errors_by_category.csv")
print("Settings tried:", len(tuning))
tuning.head(5)

Settings tried: 12


,cv_rows,min_samples_leaf,max_leaf_nodes,max_iter,learning_rate,l2_regularization,cv_rmse_mean,cv_rmse_std,fit_time_s
0,200000,100,31,400,0.1000,0.0000,0.4510,0.0025,2.9515
1,200000,100,31,600,0.0500,1.0000,0.4511,0.0024,4.2789
2,200000,100,63,600,0.0300,0.1000,0.4511,0.0025,6.2058
3,200000,200,127,200,0.0500,1.0000,0.4512,0.0025,4.7479
4,200000,200,15,400,0.0500,0.1000,0.4512,0.0025,4.7185


In [9]:
# How much did tuning help compared with the default settings from section 4?
default = cv_results.set_index("model").loc["HistGradientBoosting"]
best = tuning.iloc[0]
print(f"Default HGB CV RMSE: {default['rmse_mean']:.4f} ± {default['rmse_std']:.4f}")
print(f"Tuned   HGB CV RMSE: {best['cv_rmse_mean']:.4f} ± {best['cv_rmse_std']:.4f}")
print(f"Spread of the top 5 settings: {tuning['cv_rmse_mean'].head(5).max() - tuning['cv_rmse_mean'].head(5).min():.4f}")

Default HGB CV RMSE: 0.4515 ± 0.0025
Tuned   HGB CV RMSE: 0.4510 ± 0.0025
Spread of the top 5 settings: 0.0002


**Reading the tuning results.** The best setting (learning rate 0.1, 400 iterations, 31 leaves per tree, at least 100 products per leaf, no extra L2 penalty) reaches a CV RMSE of **0.4510 ± 0.0025**, compared with **0.4515 ± 0.0025** for the default settings. The top five settings are all within **0.0002** of each other, much less than the fold-to-fold spread. So tuning made almost no difference: HistGradientBoosting's defaults were already close to the best it can do with these features. **The limit on accuracy is the information in the features, not the model settings.** We keep the tuned setting because it was chosen by a fair search, not because it is meaningfully better.

## 7. Final evaluation on the held-out test set

`evaluate_on_test()` fits the tuned model on **all 940,325 training products** and scores it **once** on the **235,082 test products** that no step so far has seen. The mean baseline is fitted and scored the same way. Train and test metrics are both shown, so overfitting would show up as a gap.

In [10]:
test_results

,model,split,rows,rmse,mae,r2
0,Baseline (mean),train,940325,0.5085,0.3946,0.0000
1,Baseline (mean),test,235082,0.5092,0.3953,-0.0000
2,HistGradientBoosting (tuned),train,940325,0.3956,0.2875,0.3949
3,HistGradientBoosting (tuned),test,235082,0.4368,0.3207,0.2642


In [11]:
# Test-set improvement of the tuned model over the baseline
t = test_results[test_results["split"] == "test"].set_index("model")
base_t = t.loc["Baseline (mean)"]
model_t = t.loc["HistGradientBoosting (tuned)"]
for m in ["rmse", "mae"]:
    print(f"Test {m.upper()}: {base_t[m]:.4f} -> {model_t[m]:.4f} stars "
          f"({100 * (1 - model_t[m] / base_t[m]):.1f}% lower)")
print(f"Test R2: {model_t['r2']:.4f}")

Test RMSE: 0.5092 -> 0.4368 stars (14.2% lower)
Test MAE: 0.3953 -> 0.3207 stars (18.9% lower)
Test R2: 0.2642


**Reading the test results.**

- **The model clearly beats the baseline on unseen products.** Test RMSE falls from **0.509** to **0.437** stars (**14.2% lower**), MAE from **0.395** to **0.321** stars (**18.9% lower**), and the model explains **26.4%** of the variation in ratings (R² 0.264).
- **The test score is better than the CV score** (0.437 vs 0.451). This is expected, not a sign of a mistake: the final model learned from 940,325 products instead of 160,000 per CV fold. The `store` feature gains the most from more data, because each store's typical rating is estimated from more of its products.
- **Train vs test gap.** Training RMSE is 0.396 (R² 0.395) against 0.437 (R² 0.264) on test, so the model fits the products it learned from noticeably better than new ones. Part of this gap is mechanical: when scoring the *training* rows, the store encoder uses store averages that include those same products, which flatters the training score. The **test** numbers are the honest estimate, and they are the ones we report.
- **What the error means in practice.** A typical miss is about **a third of a star** (MAE 0.32). On a scale where the middle half of products sits between 4.0 and 4.7 stars (notebook 03), the model can tell a weak product from a strong one, but cannot rank products that are a few tenths of a star apart.

## 8. Which features matter

`importance_table()` uses **permutation importance** on a 20,000-product sample of the test set: shuffle one feature at a time, re-score, and measure how much RMSE gets worse. A bigger increase means the model depends on that feature more. Because it shuffles the original columns, each of the 12 features gets one score, including `store`, which the preprocessor encodes internally.

In [12]:
importance

,feature,rmse_increase_mean,rmse_increase_std,rows
0,store,0.0772,0.0013,20000
1,log_rating_number,0.0180,0.0005,20000
2,has_price,0.0056,0.0002,20000
3,title_len,0.0034,0.0001,20000
4,main_category,0.0022,0.0002,20000
5,n_description,0.0019,0.0003,20000
6,category,0.0016,0.0002,20000
7,n_details,0.0014,0.0001,20000
8,log_price,0.0013,0.0001,20000
9,n_categories,0.0012,0.0002,20000


**Reading the importance table.**

- **`store` dominates.** Shuffling it worsens RMSE by **0.077** stars, more than four times any other feature. Who sells a product (its brand or seller, encoded as that store's typical rating in the *training* data) is the strongest signal of how it will be rated. This is not target leakage: the encoder is learned only from training products, with internal cross-fitting, and the test products' own ratings are never used.
- **Review volume is second** (`log_rating_number`, **0.018**): products with more ratings tend to rate higher, which matches the EDA (notebook 03).
- **Whether a price is listed matters more than the price itself** (`has_price` **0.0056** vs `log_price` **0.0013**). The EDA showed that unpriced listings are less established and rate lower.
- **Listing detail adds a little** (`title_len` 0.0034, `n_description` 0.0019, `n_details` 0.0014, `n_categories` 0.0012, `n_features` 0.0009).
- **Category features look small** (`main_category` 0.0022, `category` 0.0016) most likely because `store` already carries much of the same information, so shuffling the category alone costs the model little.
- **`price_is_from` contributes nothing** (0.0000) and could be dropped.

## 9. Where the model is weakest

Test errors per source category, from the saved predictions (`test_predictions.csv`, which Person 4's charts also use). `mean_residual` is the average of actual minus predicted: near zero means the model is not systematically too high or too low for that category.

In [13]:
by_category

,category,rows,rmse,mae,mean_residual,mean_actual
0,All_Beauty,15321,0.5098,0.3927,0.0029,3.9454
1,Musical_Instruments,27442,0.4504,0.3391,-0.0035,4.2187
2,Industrial_and_Scientific,52703,0.4494,0.3335,-0.0017,4.2684
3,Toys_and_Games,139616,0.4203,0.3043,0.0006,4.3221


**Reading the category errors.** The model is weakest on **All_Beauty** (test RMSE **0.510**) and strongest on **Toys_and_Games** (**0.420**), with Musical_Instruments (0.450) and Industrial_and_Scientific (0.449) in between. All_Beauty is also the lowest-rated category (mean **3.95** stars vs 4.22–4.32 for the others) and, from the EDA, the one with the widest spread, so its ratings are simply harder to predict from the listing. Mean residuals are all within **±0.004** stars of zero, so the model is **not biased** against any category; it is just less precise for Beauty.

## 10. Summary of findings and limitations

**Findings** (every number from `reports/tables/`):

1. Listing characteristics predict ratings better than guessing the average: tuned HistGradientBoosting cuts test RMSE by **14.2%** (0.509 → 0.437 stars) and explains **26.4%** of the variation (`test_results.csv`).
2. The seller matters most. `store` is by far the most important feature, followed by review volume and whether a price is listed. The price level itself has little effect (`feature_importance.csv`).
3. Non-linear models do better than a linear one (CV RMSE 0.452 vs 0.460 for Ridge), and tuning adds almost nothing on top (`cv_results.csv`, `tuning_results.csv`).
4. Predictions are least precise for All_Beauty (RMSE 0.510 vs 0.420 for Toys), the lowest-rated and most variable category, with no systematic bias in any category (`test_errors_by_category.csv`).

**Limitations of the modeling:**

- About three quarters of the variation in ratings is not explained. Product quality, shipping, price relative to competitors and customer expectations are not in the product metadata.
- The `store` effect describes **association, not cause**: a store's past ratings predict its new products' ratings, but the model cannot say *why*.
- Products with fewer than 5 ratings were excluded (Person 2's `MIN_RATINGS`), so the model does not describe brand-new or niche products. A new store with no history gets an average store value.
- CV and tuning used a 200,000-row stratified sample for run time. The final model and the test evaluation use the full data.
- Training scores are optimistic because of how the store encoder treats rows it has seen. Only test scores are reported as performance.